# DQT Quickstart

Five steps with the bundled `sample_orders.csv` dataset:
`profile` → `assess` → `export` → review artifacts → `compare`

Run from repo root or open directly in Jupyter — the setup cell handles the working directory.

In [ ]:
import os, subprocess, sys, json, pathlib

# Set working directory to repo root (works whether Jupyter opens from examples/ or repo root)
_cwd = pathlib.Path(os.path.abspath(""))
REPO_ROOT = _cwd.parent if _cwd.name == "examples" else _cwd
os.chdir(REPO_ROOT)

DATASET = "examples/demo/sample_orders.csv"
OUTDIR = "dist/quickstart"

def dqt(*args):
    r = subprocess.run(
        [sys.executable, "-m", "data_quality_toolkit.cli.main", *args],
        capture_output=True, text=True
    )
    return r.stdout

print(dqt("version").strip())

In [ ]:
# Step 1 — Profile
out = json.loads(dqt("profile", DATASET))
p = out["profile"]
print(f"Rows: {p['rows']}  Cols: {p['cols']}")
print(f"Columns with nulls: {[c['name'] for c in p['columns'] if c['nulls'] > 0]}")

In [ ]:
# Step 2 — Assess
out = json.loads(dqt("assess", DATASET))
a = out["assessment"]
print(f"Score: {a['score']:.2%}  Issues: {len(a['issues'])}")
for issue in a["issues"]:
    print(f"  {issue['column']}: {issue['type']} ({issue['severity']})")

In [ ]:
# Step 3 — Export star-schema artifacts
dqt("export", DATASET, "--outdir", OUTDIR)

report = json.loads(pathlib.Path(OUTDIR, "star", "quality_report.json").read_text())
print(f"Score: {report['score']:.2%}  Issues: {report['issues_total']}")
print(f"Artifacts: {list(report['artifacts'].keys())}")

In [ ]:
# Step 4 — Compare runs (export twice, then compare)
dqt("export", DATASET, "--outdir", OUTDIR)
out = json.loads(dqt("compare", DATASET, "--outdir", OUTDIR))
print(f"Score:  {out['previous_score']:.2%} -> {out['current_score']:.2%}  (delta {out['score_delta']:+.4f})")
print(f"Issues: {out['previous_issues_total']} -> {out['current_issues_total']}")

## Next steps

- Replace `examples/demo/sample_orders.csv` with your own CSV and re-run
- Open `dist/quickstart/star/quality_report.json` to review score and issue details
- Load `fact_issues.csv` and `fact_quality_metrics.csv` into Power BI or any BI tool
- See [docs/demo_story.md](../docs/demo_story.md) for a full narrative walkthrough
- See [examples/demo/issue_showcase/README.md](demo/issue_showcase/README.md) for all detectable issue types